# Mamba-2 Language Model demo

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import time

import torch
from transformers import AutoTokenizer

from mamba2 import Mamba2LMHeadModel

if torch.cuda.is_available():
    device = torch.device('cuda:1')  # Use GPU 1
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

Official pretrained models on [huggingface](https://huggingface.co/state-spaces):
* `state-spaces/mamba2-130m`
* `state-spaces/mamba2-370m`
* `state-spaces/mamba2-780m`
* `state-spaces/mamba2-1.3b`
* `state-spaces/mamba2-2.7b`

Choose a model depending on available system RAM (for CPU or system with unified memory) or VRAM.

Note that these are base models without fine-tuning for downstream tasks such as chat or instruction following.

In [5]:
model = Mamba2LMHeadModel.from_pretrained("state-spaces/mamba2-1.3b", device=device)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token_id = tokenizer.eos_token_id

In [6]:
generation_config = dict(
    max_new_length=200,
    temperature=1.0,
    top_k=30,
    top_p=1.0,
)

In [7]:
def generate(prompt: str, seed: int = 0, show_perf: bool = True):
    """Generate streaming completion"""
    torch.manual_seed(seed)

    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)[0]
    print(prompt, end="")

    start = time.process_time()
    n_generated = 0
    for i, (token_id, _hidden_state) in enumerate(model.generate(input_ids, **generation_config)):
        token = tokenizer.decode([token_id])
        if i == 0:
            now = time.process_time()
            prompt_eval_elapsed, start = now - start, now
        else:
            n_generated += 1
        print(token, end="", flush=True)
    if show_perf:
        elapsed = time.process_time() - start
        print('\n\n---')
        print(f'Prompt eval | tokens: {input_ids.shape[0]} | elapsed: {prompt_eval_elapsed:.2f}s | tok/s: {input_ids.shape[0] / prompt_eval_elapsed:.2f}')
        print(f'Generation | tokens: {n_generated} | elapsed: {elapsed:.2f}s | tok/s: {n_generated / elapsed:.2f}')

In [8]:
generate("Mamba is a new state space model architecture")

Mamba is a new state space model architecture, for applications such as neural signal processing and classification. It has been implemented as a C library for Windows.

Features are:

Vector autoregressive models of any order

Bayesian state space models of any order

SVM models of any order

K-Means clustering models

Support for linear transformations, including scaling, translation, rotation, and scaling+translation

Support for random data as a distribution of samples for both the training and testing

Support for non-linear transformation models by the choice of activation function through a weighted least squares cost

The Mamba library can be downloaded at the following URL: https://github.com/louisdal/mamba

---
Prompt eval | tokens: 9 | elapsed: 1.33s | tok/s: 6.77
Generation | tokens: 144 | elapsed: 4.62s | tok/s: 31.17


In [9]:
generate("The meaning of life is")

The meaning of life is one that you choose. And this is what I want to tell you, as a person who has been through so much, and whose life will go on no matter what.

---
Prompt eval | tokens: 5 | elapsed: 0.18s | tok/s: 27.55
Generation | tokens: 34 | elapsed: 1.11s | tok/s: 30.54


In [10]:
generate("CUDA is Nvidia's biggest moat")

CUDA is Nvidia's biggest moat, but you can build a strong case for it even without it. If you're making high-end gaming PC (Gigabytes of RAM, beefy graphics cards, beefy cooling systems).

Nvidia's GPUs are the most powerful, reliable, and expensive parts in the industry. GPUs are very power hungry, so if they run hot, things can get complicated really fast (I learned this by the ways of my Razer Core. A lot!).

If you're looking to build a gaming PC or something that needs lots of RAM, you can build a PC with a huge amount of RAM, but most people use them like me. Most of the times, you can get away with 8GB RAM.

Then your graphics cards are your largest financial investment and your biggest power wasters. A good GPU can cost a few grand. But as Nvidia makes more and more powerful GPUs, the price comes down. It's hard to build a

---
Prompt eval | tokens: 9 | elapsed: 0.35s | tok/s: 25.97
Generation | tokens: 199 | elapsed: 6.33s | tok/s: 31.43


In [11]:
# Import Memory Caching variant
from mamba2_mc import Mamba2MCLMHeadModel

In [12]:
# Initialize MC model from pretrained weights
# MC params are freshly initialized; only base Mamba-2 params are loaded
model_mc = Mamba2MCLMHeadModel.from_pretrained(
    "state-spaces/mamba2-1.3b",
    device=device,
    segment_size=256,
    d_pool=64,
    top_k_ssc=0,  # 0 = GRM (attend all cached segments)
)

# Put both models in eval mode
model.eval()
model_mc.eval()

Mamba2MCLMHeadModel(
  (backbone): ModuleDict(
    (embedding): Embedding(50288, 2048)
    (layers): ModuleList(
      (0-47): 48 x ModuleDict(
        (mixer): Mamba2MC(
          (in_proj): Linear(in_features=2048, out_features=8512, bias=False)
          (conv1d): Conv1d(4352, 4352, kernel_size=(4,), stride=(1,), padding=(3,), groups=4352)
          (norm): RMSNorm()
          (out_proj): Linear(in_features=4096, out_features=2048, bias=False)
          (W_u): Linear(in_features=2048, out_features=64, bias=False)
        )
        (norm): RMSNorm()
      )
    )
    (norm_f): RMSNorm()
  )
  (lm_head): Linear(in_features=2048, out_features=50288, bias=False)
)

In [13]:
def generate_mc(prompt: str, seed: int = 0, show_perf: bool = True):
    """Generate streaming completion using Memory Caching variant"""
    torch.manual_seed(seed)

    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)[0]
    print(prompt, end="")

    start = time.process_time()
    n_generated = 0
    for i, (token_id, _hidden_state) in enumerate(model_mc.generate(input_ids, **generation_config)):
        token = tokenizer.decode([token_id])
        if i == 0:
            now = time.process_time()
            prompt_eval_elapsed, start = now - start, now
        else:
            n_generated += 1
        print(token, end="", flush=True)
    if show_perf:
        elapsed = time.process_time() - start
        print('\n\n---')
        print(f'Prompt eval | tokens: {input_ids.shape[0]} | elapsed: {prompt_eval_elapsed:.2f}s | tok/s: {input_ids.shape[0] / prompt_eval_elapsed:.2f}')
        print(f'Generation | tokens: {n_generated} | elapsed: {elapsed:.2f}s | tok/s: {n_generated / elapsed:.2f}')

## Memory Caching (MC) Comparison

Compare vanilla Mamba-2 with the Memory Caching variant from [arXiv:2602.24281](https://arxiv.org/pdf/2602.24281).
The MC variant caches segment states and uses gated residual memory (GRM) to retrieve relevant cached segments during generation.


In [14]:
prompt = "Mamba is a new state space model architecture"
print("=== Vanilla Mamba-2 ===")
generate(prompt, seed=0)

=== Vanilla Mamba-2 ===
Mamba is a new state space model architecture, for applications such as neural signal processing and classification. It has been implemented as a C library for Windows.

Features are:

Vector autoregressive models of any order

Bayesian state space models of any order

SVM models of any order

K-Means clustering models

Support for linear transformations, including scaling, translation, rotation, and scaling+translation

Support for random data as a distribution of samples for both the training and testing

Support for non-linear transformation models by the choice of activation function through a weighted least squares cost

The Mamba library can be downloaded at the following URL: https://github.com/louisdal/mamba

---
Prompt eval | tokens: 9 | elapsed: 0.36s | tok/s: 25.05
Generation | tokens: 144 | elapsed: 5.26s | tok/s: 27.38


In [15]:
print("\n=== Mamba-2 with Memory Caching (MC) ===")
generate_mc(prompt, seed=0)


=== Mamba-2 with Memory Caching (MC) ===
Mamba is a new state space model architecture, for applications such as neural signal processing and classification. It has been implemented as a C library for Windows.

Features are:

Vector autoregressive models of any order

Bayesian state space models of any order

SVM models of any order

K-Means clustering models

Support for linear transformations, including scaling, translation, rotation, and scaling+translation

Support for random data as a distribution of samples for both the training and testing

Support for non-linear transformation models by the choice of activation function through a weighted least squares cost

The Mamba library can be downloaded at the following URL: https://github.com/louisdal/mamba

---
Prompt eval | tokens: 9 | elapsed: 0.28s | tok/s: 32.20
Generation | tokens: 144 | elapsed: 5.55s | tok/s: 25.97


In [16]:
prompt2 = "The meaning of life is"
print("\n" + "="*50)
print("=== Vanilla Mamba-2 ===")
generate(prompt2, seed=42)


=== Vanilla Mamba-2 ===
The meaning of life is not about what happens but about what you believe it does.

Thursday, 27 September 2013

What are the best ways to make a contribution ?

I am writing this while sitting on the floor of a room at the office of the International Council for a Christian and Missionary Alliance, an organisation that I work for – which may not sound very significant. On the contrary, however, what I have to say is that all people, Christians, Muslims or others, have the ability to make a difference. And this is especially true for those of us working in the field in which we live and which involves trying to bring people into the Church so that the Holy Spirit is at work – and can be a blessing – in the lives of others.

One of the main reasons for my work in the field of Missional Christianity is to encourage people to be more self-giving in their efforts to make a difference, both within the Church as well as with those outside the Church.

---
Prompt eval 

In [17]:
print("\n=== Mamba-2 with Memory Caching (MC) ===")
generate_mc(prompt2, seed=42)


=== Mamba-2 with Memory Caching (MC) ===
The meaning of life is not about what happens but about what you believe it does.

Thursday, 27 September 2013

What are the best ways to make a contribution ?

I am writing this while sitting on the floor of a room at the office of the International Council for a Christian and Missionary Alliance, an organisation that I work for – which may not sound very significant. On the contrary, however, what I have to say is that all people, Christians, Muslims or others, have the ability to make a difference. And this is especially true for those of us working in the field in which we live and which involves trying to bring people into the Church so that the Holy Spirit is at work – and can be a blessing – in the lives of others.

One of the main reasons for my work in the field of Missional Christianity is to encourage people to be more self-giving in their efforts to make a difference, both within the Church as well as with those outside the Church.
